### 1. Imports

In [13]:
import os
import requests
from typing import TypedDict

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END

### 2. Configurations

In [14]:
with open(r"C:\Users\user\Desktop\sharad-ai\shared\ey-ai-upskill-13\key-vault\ey-key\api.key") as f:
    openai_api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = openai_api_key

In [15]:
CIS_URL = r"http://localhost:7860/api/v1/run/eef87fb7-0ffe-4987-9be8-2e17813b7eb0"

In [16]:
with open(r"C:\Users\user\Desktop\sharad-ai\shared\ey-ai-upskill-13\key-vault\NVE\api.key") as f:
    nvd_api_key = f.read().strip()
NVD_API_KEY = nvd_api_key

In [17]:
MODEL = "gpt-4.1-mini"
llm = ChatOpenAI(model=MODEL, temperature=0)

### 3. Build the Tools

#### CIS Tool  - NOT USED AS USING LOCAL RAG 

In [ ]:
import os

import requests

url = "http://localhost:7860/api/v2/workflows"
payload = {
    "flow_id": "eef87fb7-0ffe-4987-9be8-2e17813b7eb0",
    "input_value": "what are the windows password requirements for a strong password?"
}
headers = {
    "Content-Type": "application/json",
    "x-api-key": "sk-qNz-70oQVfAE8wvyRadRAVpwVDbfBtxcj_ivZNHx_Ow",
}

response = requests.post(url, json=payload, headers=headers)
response.raise_for_status()
result = response.json()

# The text reply is in output.text:
print(result["output"]["text"])

In [ ]:
import requests
from langchain_core.tools import tool


@tool
def ask_cis(question: str) -> str:
    """Query the Flowise workflow knowledge base for CIS benchmarks and security guidance.

    Args:
        question: The security question or topic to search.
    """
    url = "http://localhost:7860/api/v2/workflows"
    headers = {
        "Content-Type": "application/json",
        "x-api-key": "sk-qNz-70oQVfAE8wvyRadRAVpwVDbfBtxcj_ivZNHx_Ow",
    }
    payload = {
        "flow_id": "eef87fb7-0ffe-4987-9be8-2e17813b7eb0",
        "input_value": question,
    }

    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        response.raise_for_status()
        result = response.json()

        # Safely extract output text
        return result.get("output", {}).get("text", "No output returned.")
    except requests.RequestException as e:
        return f"Error executing CIS workflow request: {e}"

In [ ]:
ask_cis.invoke("Tell me about proper shutdown of windows servers")

#### NVD Tool

In [18]:
@tool
def lookup_cve(keyword:str)->str:
    """Query the NVD API for CVEs matching a keyword."""
    print("[TOOL]CVE Tool Activated")
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    params = {"keywordSearch": keyword, "resultsPerPage": 3}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        data = r.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return "No CVEs found."
        lines = []
        for v in vulns:
            c = v["cve"]
            lines.append(f"{c['id']}: {c['descriptions'][0]['value'][:180]}")
        return "\n".join(lines)
    except Exception as e:
        return f"CVE lookup failed: {e}"

In [19]:
lookup_cve.invoke("SMB")

[TOOL]CVE Tool Activated


'CVE-1999-1387: Windows NT 4.0 SP2 allows remote attackers to cause a denial of service (crash), possibly via malformed inputs or packets, such as those generated by a Linux smbmount command that \nCVE-1999-0225: Windows NT 4.0 allows remote attackers to cause a denial of service via a malformed SMB logon request in which the actual data size does not match the specified size.\nCVE-1999-0495: A remote attacker can gain access to a file system using ..  (dot dot) when accessing SMB shares.'

### ask_cis_local localhost:8000/ask

In [51]:

import requests
from langchain_core.tools import tool


API_URL = "http://localhost:8000/ask"


@tool
def ask_cis_local(query: str) -> str:
    """Query the locally running knowledge base for CIS benchmarks and security guidance.

    Args:
        question: The security question or topic to search.
    """
    try:
        response = requests.post(
            API_URL,
            json={"query": query},
            timeout=60,
        )

        response.raise_for_status()
        data = response.json()

        return data["answer"]

    except requests.exceptions.Timeout:
        return "Error: The local assistant API request timed out."

    except requests.exceptions.ConnectionError:
        return (
            "Error: Could not connect to the local assistant API. "
            "Ensure the FastAPI server is running on port 8080."
        )

    except requests.exceptions.HTTPError as e:
        return f"API HTTP error: {e.response.text}"

    except (ValueError, KeyError) as e:
        return f"Invalid API response: {e}"


In [52]:
ask_cis_local.invoke("Tell me about proper shutdown of windows servers")

'1. Answer: Proper shutdown of Windows servers involves using designated shutdown methods to ensure that all applications and services are closed gracefully. This prevents data loss, corruption, and maintains system integrity. The recommended approach is to use the "Shut Down" option from the Start menu or the command line, ensuring that users are notified and critical processes are completed before shutdown.\n\n2. Relevant CIS Controls:\n   - 18.9.21 Ensure \'Shut down the system\' is set to \'Administrators, Users\'\n   - 18.9.22 Ensure \'Force shutdown from a remote system\' is set to \'Administrators\'\n\n3. Summary: Proper shutdown procedures are essential for maintaining system integrity and preventing data loss on Windows servers. Using the appropriate shutdown methods ensures that all applications and services terminate correctly, reducing the risk of corruption and facilitating a smooth restart.\n\n4. Recommendations:\n   - Always initiate shutdown using the "Shut Down" option

#### IP Config Tool

In [22]:
import subprocess

@tool
def ipconfig_tool() -> str:
    """
    Returns the Windows network configuration using the ipconfig command.
    """
    try:
        result = subprocess.run(
            ["ipconfig"],
            capture_output=True,
            text=True,
            check=True,
            shell=True
        )

        return result.stdout

    except subprocess.CalledProcessError as e:
        return f"Error executing ipconfig:\n{e.stderr}"

In [23]:
print(ipconfig_tool.invoke({}))


Windows IP Configuration


Ethernet adapter Ethernet0:

   Connection-specific DNS Suffix  . : deepcloud.in
   Link-local IPv6 Address . . . . . : fe80::ec70:332e:cf53:90e4%5
   IPv4 Address. . . . . . . . . . . : 10.33.153.54
   Subnet Mask . . . . . . . . . . . : 255.255.255.0
   Default Gateway . . . . . . . . . : 10.33.153.1



### 4. Agent Layer

In [24]:
planner = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a planner. Decide weather RAG and CVE lookup is required"
)

In [27]:
retrieval_agent = create_agent(
    model=llm,
    tools=[ask_cis_local],
    system_prompt="Use the CIS policy tool to retrieve CIS benchmark guidance. Use the provided tools mandatorily"
)

In [28]:
threat_agent = create_agent(
    model=llm,
    tools=[lookup_cve],
    system_prompt="Use CVE lookup tool when threat intelligence is needed. Use the provided tools mandatorily"
)

In [29]:
validator_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Validate the evidence and produce a concise final answer"
)

In [30]:
response = planner.invoke({ 'messages': 'what is the password requriement for windows?'})

In [31]:
print(response['messages'][1].content)

For the query about password requirements for Windows, a CVE (Common Vulnerabilities and Exposures) lookup is not required because this is a general configuration or policy question rather than a security vulnerability inquiry. However, RAG (Retrieval-Augmented Generation) could be useful to retrieve the most current and detailed password policy information from official Microsoft documentation or trusted sources.

Decision:
- RAG: Required
- CVE lookup: Not required


### 5. State

In [32]:
class CyberState(TypedDict):
    query: str
    plan: str
    rag: str
    cves: str
    final: str

### 6. Nodes

In [33]:
def planner_node(state: CyberState):
    r = planner.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"plan": str(r)}

In [34]:
def rag_node(state: CyberState):
    r = retrieval_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"rag": str(r)}

In [35]:
def cve_node(state: CyberState):
    r = threat_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"cves": str(r)}

In [36]:
def validator_node(state: CyberState):

    prompt = f"""
User Query:
{state["query"]}

Plan:
{state.get("plan", "")}

RAG:
{state.get("rag", "")}

CVEs:
{state.get("cves", "")}

SCORE:
validation score

Produce the final validated response.
Also, give a score from 0 to 10
"""
    r = validator_agent.invoke({
        "messages":[
            {"role":"user", "content":prompt}
        ]
    })
    return {"final": r}

### 7. Workflow

In [37]:
graph = StateGraph(CyberState)

graph.add_node("planner", planner_node)
graph.add_node("rag", rag_node)
graph.add_node("cve", cve_node)
graph.add_node("validator", validator_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "rag")
graph.add_edge("rag", "cve")
graph.add_edge("cve", "validator")
graph.add_edge("validator", END)

In [38]:
app = graph.compile()

### 8. Execute

In [39]:
result = app.invoke({
    "query": "How can I harden Windows SMB services against ransomware?"
})

In [40]:
result

{'query': 'How can I harden Windows SMB services against ransomware?',
 'plan': "{'messages': [HumanMessage(content='How can I harden Windows SMB services against ransomware?', additional_kwargs={}, response_metadata={}, id='0b0aa660-7a76-4963-9bb9-a2d44d416a8f'), AIMessage(content='RAG (Retrieval-Augmented Generation) and CVE (Common Vulnerabilities and Exposures) lookup would be helpful here to provide the most up-to-date and specific security measures and known vulnerabilities related to Windows SMB services.\\n\\nI will proceed with RAG and CVE lookup to gather detailed and current information on hardening Windows SMB services against ransomware.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 37, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tok

In [41]:
print(result["final"]["messages"][-1].content)

Final Answer:

To harden Windows SMB services against ransomware, follow these key steps aligned with CIS benchmarks and security best practices:

1. **Disable SMBv1:** SMBv1 is outdated and vulnerable, frequently exploited by ransomware. Disable it via Group Policy:
   - Path: `Computer Configuration\Policies\Administrative Templates\MS Security Guide\Configure SMB v1 server`
   - Set to: **Disabled**

2. **Mandate SMBv3 (minimum version 3.1.1):** SMBv3 offers enhanced security features including encryption and improved authentication.
   - Group Policy setting: **Mandate the minimum version of SMB**
   - Set to: **Enabled: 3.1.1**

3. **Require SMB Encryption:** Encrypt SMB traffic to protect data in transit from interception or tampering.
   - Group Policy setting: **Require Encryption**
   - Set to: **Enabled**

4. **Apply Security Patches Promptly:** Keep Windows systems updated with the latest patches to fix SMB vulnerabilities.

5. **Restrict SMB Access:** Limit SMB access to on

In [42]:
result = app.invoke({
    "query": "How can I secure windows SMB against latest attacks?"
})

[TOOL]CVE Tool Activated


In [43]:
result

{'query': 'How can I secure windows SMB against latest attacks?',
 'plan': "{'messages': [HumanMessage(content='How can I secure windows SMB against latest attacks?', additional_kwargs={}, response_metadata={}, id='418bee25-3efe-47a8-b371-48ac19ef0e8d'), AIMessage(content='For securing Windows SMB against the latest attacks, a Retrieval-Augmented Generation (RAG) approach can be helpful to fetch the most recent and detailed security guidelines. Additionally, a Common Vulnerabilities and Exposures (CVE) lookup is essential to identify the latest vulnerabilities affecting SMB and apply relevant patches or mitigations.\\n\\nTherefore, both RAG and CVE lookup are required.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 36, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 

In [44]:
print(result["final"]["messages"][-1].content)

Final Answer:

To secure Windows SMB against the latest attacks, follow these validated best practices based on current CIS benchmarks and known vulnerabilities:

1. **Disable SMBv1:**  
   SMBv1 is outdated and vulnerable. Disable it via Group Policy:  
   `Computer Configuration\Policies\Administrative Templates\MS Security Guide\Configure SMB v1 server` → set to **Disabled**.

2. **Mandate SMBv3 (minimum version 3.1.1):**  
   Enforce use of SMBv3 for improved security by enabling:  
   `Mandate the minimum version of SMB` → set to **Enabled: 3.1.1**.

3. **Require Encryption for SMB Traffic:**  
   Enable encryption to protect data in transit:  
   Set `Require Encryption` to **Enabled** in Group Policy.

4. **Enable SMB Signing and Auditing:**  
   - Enable auditing for servers that do not support encryption and signing to detect potential vulnerabilities:  
     - `Audit server does not support encryption` → Enabled  
     - `Audit server does not support signing` → Enabled  
   

In [45]:
Tool 3:

SyntaxError: invalid syntax (1715492085.py, line 1)

In [ ]:
Tool 4

In [46]:
Tool 5

SyntaxError: invalid syntax (3022942940.py, line 1)

In [49]:
tools = ["ask_cis_local", "lookup_cve", "lookup_cisa_kev", "lookup_attack", "lookup_epss"]
cyber_agent = create_agent(llm, tools=tools, system_prompt="write_your_prompt")

AttributeError: 'function' object has no attribute 'name'

In [50]:
cyber_agent.invoke(m1)

NameError: name 'cyber_agent' is not defined